# RQ4 (E10): labeled context and the value of CPT

Compare original TabPFN v3 and TabICLv2 with a fixed continued-pretraining (CPT) recipe as labeled context shrinks. Each target table was excluded from its CPT checkpoint. No weights are fitted here. Five outer folds cover all test rows; a separate, fixed validation-label budget supports threshold and calibration rules. This is a context-memory curve, not a curve for the total number of labels acquired.

Does credit CPT change the value of labeled context on an excluded credit table? Dependence on context size is different from variability across repeated contexts of the same size.


## 0. Evidence and scope

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
from src.visualize import context_curves as cc
sink = FigureSaver('rq4/01_context_curves')
report = cp.NotebookReport('RQ4 (E10): labeled context and the value of CPT', sink=sink)
print(source_description())
report.add('0. Evidence and scope', source_description() + '\n' + 'Does credit CPT change the value of labeled context on an excluded credit table? Dependence on context size is different from variability across repeated contexts of the same size.')
study = cc.load_context_study()

## 1. Fixed checkpoints and evaluation protocol

The CPT reference is the predefined full-parameter experiment-1 recipe, not the best result observed in the sweep. Context prefixes are nested and shared between original and CPT models. Inference views stay fixed within each family.

In [ ]:
report.table('Evaluation protocol', cc.protocol_table(study))
report.add('1. Fixed checkpoints and evaluation protocol', 'The 50,000-row cap bounds computation; it is not TabPFN v3’s native limit. The available-pool endpoint is included when distinct. Validation and test rows never enter context.')

## 2. Completed exports and five-fold support

A completed task supplies one table, model family and outer fold. Its compressed metric export must match the prepared plan and completion receipt. Partially written SQLite journals are recovery evidence and are not counted as completed metric exports.

In [ ]:
report.table('Task inventory', study.inventory)
cp.show(sink, cc.plot_context_support(study))
report.add('2. Completed exports and five-fold support', 'Absent exports are unavailable evidence, not zero performance. DATA plans/receipts and project metric exports are both needed. Coverage counts are not live scheduler status.')

## 3. Original and CPT context curves

Separate panels cover context horizons of 128, 2,048 and 50,000 rows. Within each horizon, the same tables must support every displayed context size and both model families. The fixed cohorts prevent changes in table mix from looking like gains from more context. Different horizons can therefore have different table cohorts. Thin gray effect curves retain individual tables; the thicker colored curve is the equal-table mean. Neither is a repeated-context variance estimate. TabICLv2 uses automatic inference; its selected-table precision diagnostic qualifies attribution of large gains.

In [ ]:
cp.show(sink, cc.plot_context_curves(study))
cp.show(sink, cc.plot_table_curves(study))
report.add('3. Original and CPT context curves', 'Panels show PD AUC/Brier and LGD R²/RMSE, with original and CPT levels and improvement-oriented paired differences. Scores first average the five paired folds within each table, then weight tables equally. Positive differences favor CPT. Individual table curves reveal opposing responses or influential tables that a mean can hide; the same table identities persist across each horizon. No figure is generated for a horizon without a complete common cohort.')

## 4. Calibration, errors and inference cost

Retain paired table-level values at every measured context size, including available-pool endpoints. These tables include proper scoring rules, validation-selected F1 and elapsed inference time; their distinct units are preserved.

In [ ]:
for metric, table in cc.secondary_tables(study).items():
    report.table(metric+' paired table values', table)
report.add('4. Calibration, errors and inference cost', 'Brier score, log loss, ECE, RMSE, MAE and elapsed-time gains use original minus CPT; AUC, R² and F1 gains use CPT minus original. Timing includes fitting and validation/test prediction, excludes prior CPT training and queue time, and can be affected by fixed execution order. Fold averages are descriptive, not independent table replications.')

## Additional TabPFN v2 and v2.6 coverage

The exploratory extension preserves the original dataset partitions and reports its evidence separately.

In [ ]:
from src.visualize import legacy_extensions
legacy_extensions.render(report, sink, 4, 'results')

## Complete text summary

All displayed numerical values and captions, in section order.

In [ ]:
print(report.summary(sink))